# Bai tap: ETHUSDs (Forex)/ ETHUSDT 1m tren Forex hoac Binace
### Mot chien luoc mua: Khi gia du doan > Gia thuc te va MA 10 > MA 20
### Mot chien luoc ban: Khi gia du doan < Gia thuc te va MA 10 < MA 20

# Lay du lieu

In [13]:
# Forex
import sys
sys.path.append('../Common')

import CommonMT5DWH
import MetaTrader5 as mt5

symbol = 'EURUSD.sml'
from_date = '1970-01-05'
to_date = '2024-12-06'
timeframe = mt5.TIMEFRAME_D1

data = CommonMT5DWH.CommonMT5DWH.loaddataMT5_FromTo_List_Ext(symbol, from_date, to_date, timeframe)

In [14]:
data

,Symbol,Datetime,Open,High,Low,Close,Volume
0,EURUSD.sml,1971-01-04,0.53690,0.53690,0.53690,0.53690,1
1,EURUSD.sml,1971-01-05,0.53660,0.53660,0.53660,0.53660,1
2,EURUSD.sml,1971-01-06,0.53650,0.53650,0.53650,0.53650,1
3,EURUSD.sml,1971-01-07,0.53680,0.53680,0.53680,0.53680,1
4,EURUSD.sml,1971-01-08,0.53710,0.53710,0.53710,0.53710,1
...,...,...,...,...,...,...,...
13895,EURUSD.sml,2024-11-29,1.05521,1.05974,1.05416,1.05779,240058
13896,EURUSD.sml,2024-12-02,1.05688,1.05709,1.04606,1.04978,280542
13897,EURUSD.sml,2024-12-03,1.04986,1.05352,1.04806,1.05095,212669
13898,EURUSD.sml,2024-12-04,1.05067,1.05443,1.04724,1.05103,231249


# Xu ly va tinh toan chien luoc

In [23]:
import talib
from sklearn.linear_model import LinearRegression 
import redis 
from datetime import datetime, timedelta
import time
import pandas as pd 
import numpy as np

# Them chi bao
data['MA10'] = talib.SMA(data['Close'], timeperiod=10)
data['MA20'] = talib.SMA(data['Close'], timeperiod=20)

# Xu ly NA: Drop NA
data = data.dropna()

# Khoi tao Feature va Target
X = data[['Open', 'High', 'Low', 'Volume', 'MA10', 'MA20']] # Feature
y = data['Close'] # Target

model = LinearRegression()
model.fit(X, y)

# Du doan
data['Predicted_Close'] = model.predict(X)

# Tinh toan Buy_Signal va Sell Signal
data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['MA10'] > data['MA20']))
data['Sell_Signal'] = ((data['Predicted_Close'] < data['Close']) & (data['MA10'] < data['MA20']))

# Day sang Redis
# Tạo kết nối Redis
r = redis.Redis(host='localhost', port=6379, db=10) # DB muon chua # CK 0-5; FX 6-10; Crypto 11-15
# Tạo hash key
hash_key = 'OG_ML_FX_MA10, MA20'

last_record = data.iloc[-1] # Lay record moi nhat

# Chuyển đổi record cuối cùng thành từ điển và lưu vào Redis dưới dạng hash
if(last_record['Buy_Signal'] == True or last_record['Sell_Signal'] == True):
	for field, value in last_record.to_dict().items():
		# Chuyển đổi giá trị uint64 và Timestamp thành chuỗi
		if isinstance(value, pd.Timestamp):
			value = value.isoformat()
		elif isinstance(value, (int, np.uint64)):
			value = str(value)
		r.hset(hash_key, field, value)  
		r.hset(hash_key, 'Symbol', symbol)
		r.hset(hash_key, 'Insertdate', datetime.now().isoformat())
		last_record['Insertdate'] = datetime.now().isoformat()
	print(last_record)   
else:
	print(last_record)   
	print('Không có tín hiệu!')

C:\Users\PCDTT\AppData\Local\Temp\ipykernel_35792\2911018036.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted_Close'] = model.predict(X)
C:\Users\PCDTT\AppData\Local\Temp\ipykernel_35792\2911018036.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Buy_Signal'] = ((data['Predicted_Close'] > data['Close']) & (data['MA10'] > data['MA20']))
C:\Users\PCDTT\AppData\Local\Temp\ipykernel_35792\2911018036.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice fr

Symbol                             EURUSD.sml
Datetime                  2024-12-05 00:00:00
Open                                  1.05112
High                                  1.05898
Low                                    1.0508
Close                                 1.05877
Volume                                 185602
MA10                                 1.052115
MA20                                 1.055281
Predicted_Close                      1.057115
Buy_Signal                              False
Sell_Signal                              True
Insertdate         2024-12-06T21:24:20.047223
Name: 13899, dtype: object


C:\Users\PCDTT\AppData\Local\Temp\ipykernel_35792\2911018036.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()
C:\Users\PCDTT\AppData\Local\Temp\ipykernel_35792\2911018036.py:49: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  last_record['Insertdate'] = datetime.now().isoformat()


In [24]:
data

,Symbol,Datetime,Open,High,Low,Close,Volume,MA10,MA20,Predicted_Close,Buy_Signal,Sell_Signal
57,EURUSD.sml,1971-03-26,0.53870,0.53870,0.53870,0.53870,1,0.538610,0.538575,0.538716,True,False
58,EURUSD.sml,1971-03-29,0.53840,0.53840,0.53840,0.53840,1,0.538610,0.538580,0.538420,True,False
59,EURUSD.sml,1971-03-30,0.53880,0.53880,0.53880,0.53880,1,0.538650,0.538590,0.538815,True,False
60,EURUSD.sml,1971-03-31,0.53870,0.53870,0.53870,0.53870,1,0.538660,0.538575,0.538717,True,False
61,EURUSD.sml,1971-04-01,0.53880,0.53880,0.53880,0.53880,1,0.538680,0.538605,0.538816,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...
13895,EURUSD.sml,2024-11-29,1.05521,1.05974,1.05416,1.05779,240058,1.053184,1.061450,1.057695,False,True
13896,EURUSD.sml,2024-12-02,1.05688,1.05709,1.04606,1.04978,280542,1.052179,1.059554,1.047596,False,True
13897,EURUSD.sml,2024-12-03,1.04986,1.05352,1.04806,1.05095,212669,1.051311,1.057452,1.051111,False,False
13898,EURUSD.sml,2024-12-04,1.05067,1.05443,1.04724,1.05103,231249,1.050977,1.056359,1.050614,False,True
